<a href="https://colab.research.google.com/github/Vanquan99/Tool_Copy-Google-Drive/blob/main/tool_coppy_driver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Copy Folder / File — Google Drive to Google Drive
Notebook dùng trên Colab để copy 1 file riêng lẻ hoặc copy toàn bộ folder giữa hai Google Drive (kể cả shared drive).

## Cách dùng nhanh
1. Mở notebook bằng Colab badge hoặc link GitHub.
2. Chạy lần lượt từng cell từ trên xuống dưới.
3. Dán link hoặc ID của folder/file nguồn và folder đích.
4. Nếu source là file, notebook sẽ copy trực tiếp file đó.

In [ ]:
#@title Input
dest_link = "" #@param {type:"string"}
source_link = "" #@param {type:"string"}
from_page = 0 #@param {type:"integer"}
to_page = 0 #@param {type:"integer"}
max_download_size_gb = 700 #@param {type:"number"}
exclude_str = "" #@param {type:"string"}
# Leave pages at 0 to include everything.

In [ ]:
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
service = build('drive', 'v3')
print('Drive service ready')

In [ ]:
import time, re

class DownloadFromDrive:
    def __init__(self, service):
        self.service = service
        self._total_size = 0.0  # MB
        self._limit_size = 0.0  # GB
        self.excluded_strings = []

    def extract_id(self, value):
        if not value:
            return None
        match = re.search(r'[-\w]{25,}', value)
        return match.group(0) if match else value

    def is_folder(self, file_meta):
        return file_meta.get('mimeType') == 'application/vnd.google-apps.folder'

    def get_children(self, folder_id, from_page=0, to_page=0):
        files = []
        page_token = None
        query = f"'{folder_id}' in parents and trashed = false"
        if self.excluded_strings:
            excluded_query = ' and '.join([f"not name contains '{item}'" for item in self.excluded_strings])
            query += f' and ({excluded_query})'
        page = 0
        while True:
            response = self.service.files().list(
                q=query,
                orderBy='name, createdTime',
                fields='files(id, name, mimeType, size), nextPageToken',
                pageToken=page_token,
                supportsAllDrives=True,
                includeItemsFromAllDrives=True
            ).execute()
            page += 1
            if (from_page < page <= to_page) or to_page == 0:
                files.extend(response.get('files', []))
            page_token = response.get('nextPageToken')
            if not page_token or (to_page > 0 and page >= to_page):
                break
        return files

    def exists_in_dest(self, dest_folder_id, name):
        try:
            escaped = name.replace("'", "\\'")
            query = f"'{dest_folder_id}' in parents and name = '{escaped}' and trashed = false"
            result = self.service.files().list(q=query, fields='files(id)').execute()
            return bool(result.get('files'))
        except Exception as exc:
            print('exists_in_dest error:', exc)
            return False

    def create_folder(self, dest_folder_id, folder_name):
        if self.exists_in_dest(dest_folder_id, folder_name):
            return self.service.files().list(
                q=f"'{dest_folder_id}' in parents and name = '{folder_name}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false",
                fields='files(id)'
            ).execute().get('files', [{}])[0].get('id')
        body = {'name': folder_name, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [dest_folder_id]}
        created = self.service.files().create(body=body, fields='id').execute()
        return created['id']

    def copy_file(self, dest_folder_id, file_meta):
        if self.exists_in_dest(dest_folder_id, file_meta['name']):
            print(f"{file_meta['name']} already exists, skipping")
            return
        start = time.time()
        self.service.files().copy(
            body={'parents': [dest_folder_id]},
            fileId=file_meta['id'],
            supportsAllDrives=True
        ).execute()
        elapsed = time.time() - start
        size_mb = int(file_meta.get('size', 0)) / (1024 * 1024)
        self._total_size += size_mb
        print(f"[{file_meta['name']}] copied {size_mb:0.2f} MB in {elapsed:0.2f}s")

    def copy_item(self, dest_folder_id, item_meta):
        if self.is_folder(item_meta):
            print(f"Entering folder: {item_meta['name']}")
            new_folder_id = self.create_folder(dest_folder_id, item_meta['name'])
            children = self.get_children(item_meta['id'], 0, 0)
            self.copy_items(new_folder_id, children)
        else:
            self.copy_file(dest_folder_id, item_meta)

    def copy_items(self, dest_folder_id, items):
        for item in items:
            self.copy_item(dest_folder_id, item)

    def copy_drive_to_drive(self, dest_link, source_link, from_page=0, to_page=0):
        dest_id = self.extract_id(dest_link)
        source_id = self.extract_id(source_link)
        if not dest_id or not source_id:
            raise ValueError('Missing destination or source ID')
        source_meta = self.service.files().get(fileId=source_id, supportsAllDrives=True, fields='id,name,mimeType,size').execute()
        if not self.is_folder(source_meta):
            print('Source is a file, copying directly')
            self.copy_file(dest_id, source_meta)
            return
        source_folder_name = source_meta['name']
        target_folder_id = self.create_folder(dest_id, source_folder_name)
        children = self.get_children(source_id, from_page, to_page)
        self.copy_items(target_folder_id, children)
        print('Done')

In [ ]:
downloader = DownloadFromDrive(service)
downloader._limit_size = float(max_download_size_gb)
downloader.excluded_strings = [item.strip() for item in exclude_str.split(',') if item.strip()]
downloader.copy_drive_to_drive(dest_link, source_link, from_page, to_page)